# 第08章：矩阵乘法（朴素版）— 从零开始理解 GEMM

## 前置知识
- 第01-07章
- 对应 CUDA 项目: `src/simt/00simt_naive.cu`

## 学习目标
- 理解矩阵乘法的计算模式和在深度学习中的核心地位
- 实现朴素版 Triton GEMM kernel
- 分析性能瓶颈：为什么朴素版很慢

In [ ]:
import torch
import triton
import triton.language as tl

## 8.1 矩阵乘法基础

$$C_{ij} = \sum_{k=0}^{K-1} A_{ik} \cdot B_{kj}$$

```
    A (M×K)           B (K×N)           C (M×N)
  ┌─────────┐      ┌─────────┐      ┌─────────┐
  │ · · · · │      │ · · · · │      │ · · · · │
M │ · · · · │  ×  K│ · · · · │  =  M│ · · · · │
  │ · · · · │      │ · · · · │      │ · · · · │
  └─────────┘      └─────────┘      └─────────┘
       K                N                 N

C[i,j] = A的第i行 · B的第j列 (点积)
```

### 计算量分析
- **FLOPs**: 2 × M × N × K（每个输出元素需要 K 次乘加）
- **内存访问**: (M×K + K×N + M×N) 个元素
- **算术强度**: 2MNK / (MK+KN+MN)  (当 M=N=K 时约为 K/3)

### 在深度学习中的地位
```
Linear Layer:    Y = X @ W + b         ← GEMM
Attention:       S = Q @ K^T            ← GEMM
                 O = softmax(S) @ V     ← GEMM  
Conv (im2col):   展开后也是 GEMM

训练一个 LLM：>90% 的计算时间在 GEMM 上！
```

## 8.2 朴素版思路

最直接的并行化：每个 program 计算 C 的一个 **BLOCK_M × BLOCK_N** 子块。

```
输出矩阵 C (M×N) 被切分为多个块:
┌──────┬──────┬──────┐
│(0,0) │(0,1) │(0,2) │  ← 每个块由一个 program 计算
├──────┼──────┼──────┤
│(1,0) │(1,1) │(1,2) │
├──────┼──────┼──────┤
│(2,0) │(2,1) │(2,2) │
└──────┴──────┴──────┘

对于 C[pid_m, pid_n] 这个块:
  需要 A 的第 pid_m 行块 (BLOCK_M × K)
  需要 B 的第 pid_n 列块 (K × BLOCK_N)

K 太大，无法一次加载，所以沿 K 维分块循环:

A:  ┌──┬──┬──┬──┐    B:  ┌──────────┐
    │k0│k1│k2│k3│        │    k0    │
    │  │  │  │  │        ├──────────┤
    └──┴──┴──┴──┘        │    k1    │
     K分成4块             ├──────────┤
                         │    k2    │
                         ├──────────┤
                         │    k3    │
                         └──────────┘

acc = 0
for k_block in [k0, k1, k2, k3]:
    acc += A_block[k_block] @ B_block[k_block]   ← tl.dot
C_block = acc
```

In [ ]:
@triton.jit
def naive_matmul_kernel(
    # 指针
    a_ptr, b_ptr, c_ptr,
    # 矩阵维度
    M, N, K,
    # 步长
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    # 块大小
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    朴素矩阵乘法: C = A @ B
    
    对应 CUDA 项目的 00simt_naive.cu 的概念，
    但已经使用了 K 维分块（因为 Triton 的 tl.dot 要求分块）。
    """
    # ============================================================
    # 第1步: 确定当前 program 负责哪个输出块
    # ============================================================
    pid_m = tl.program_id(0)  # 行方向的块索引
    pid_n = tl.program_id(1)  # 列方向的块索引
    
    # ============================================================
    # 第2步: 初始化 FP32 累加器
    # ============================================================
    # 输入是 FP16 但累加用 FP32，防止精度损失
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    # ============================================================
    # 第3步: 沿 K 维循环，每次处理 BLOCK_K 列/行
    # ============================================================
    for k in range(0, K, BLOCK_K):
        # --- 加载 A 的子块 (BLOCK_M × BLOCK_K) ---
        a_row = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # 行索引
        a_col = k + tl.arange(0, BLOCK_K)                 # K维索引
        a_ptrs = a_ptr + a_row[:, None] * stride_am + a_col[None, :] * stride_ak
        a_mask = (a_row[:, None] < M) & (a_col[None, :] < K)
        a_block = tl.load(a_ptrs, mask=a_mask, other=0.0)
        
        # --- 加载 B 的子块 (BLOCK_K × BLOCK_N) ---
        b_row = k + tl.arange(0, BLOCK_K)                 # K维索引
        b_col = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # 列索引
        b_ptrs = b_ptr + b_row[:, None] * stride_bk + b_col[None, :] * stride_bn
        b_mask = (b_row[:, None] < K) & (b_col[None, :] < N)
        b_block = tl.load(b_ptrs, mask=b_mask, other=0.0)
        
        # --- 块级矩阵乘并累加 ---
        # tl.dot: (BLOCK_M, BLOCK_K) @ (BLOCK_K, BLOCK_N) → (BLOCK_M, BLOCK_N)
        acc += tl.dot(a_block, b_block)
    
    # ============================================================
    # 第4步: 将结果写回全局内存
    # ============================================================
    c_row = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    c_col = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + c_row[:, None] * stride_cm + c_col[None, :] * stride_cn
    c_mask = (c_row[:, None] < M) & (c_col[None, :] < N)
    # 将 FP32 累加器转回 FP16 存储
    tl.store(c_ptrs, acc.to(tl.float16), mask=c_mask)

In [ ]:
def naive_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """朴素 GEMM 包装函数"""
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty(M, N, device=a.device, dtype=torch.float16)
    
    BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    
    naive_matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c

# 正确性测试
M, N, K = 512, 512, 256
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

c_triton = naive_matmul(a, b)
c_torch = (a.float() @ b.float()).half()

print(f"朴素 GEMM:")
print(f"  形状: ({M},{K}) @ ({K},{N}) = ({M},{N})")
print(f"  正确性: {torch.allclose(c_triton, c_torch, atol=1e-1)}")
print(f"  最大误差: {(c_triton.float() - c_torch.float()).abs().max().item():.4f}")

## 8.3 性能分析 — 为什么朴素版慢？

```
问题：全局内存带宽是瓶颈

每个输出块 C[i,j] (BLOCK_M × BLOCK_N) 需要:
  加载 A: BLOCK_M × K 个元素     ← 每个 A 行被加载了 N/BLOCK_N 次！
  加载 B: K × BLOCK_N 个元素     ← 每个 B 列被加载了 M/BLOCK_M 次！

A 总共被加载: M × K × (N/BLOCK_N) 次   ← 大量冗余!
B 总共被加载: K × N × (M/BLOCK_M) 次   ← 大量冗余!

对比 CUDA 00simt_naive.cu:
  CUDA 版本: 每个线程处理 1 个元素, ~6.80 ms (Blackwell)
  Triton 朴素版: 已经有分块, 但没有高级优化
```

### 优化方向预览
```
朴素版问题              → 优化方向              → 对应章节
──────────────────────────────────────────────────────────
重复加载全局内存        → 共享内存缓存          → 第12章
L2 缓存命中率低        → Swizzle 块调度        → 第13章
计算和访存没有重叠      → 软件流水线             → 第14章
没有用 Tensor Core     → tl.dot 自动映射       → 第17章
块大小固定             → 自动调优               → 第11章
```

In [ ]:
# 性能基准测试
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

ms_naive = triton.testing.do_bench(lambda: naive_matmul(a, b))
ms_torch = triton.testing.do_bench(lambda: torch.matmul(a, b))

flops = 2 * M * N * K
print(f"M={M}, N={N}, K={K} (FP16)")
print(f"{'方法':<20} {'时间(ms)':<12} {'TFLOPS':<10}")
print(f"{'─'*42}")
print(f"{'Triton 朴素版':<20} {ms_naive:<12.3f} {flops/ms_naive*1e-9:<10.1f}")
print(f"{'torch.matmul':<20} {ms_torch:<12.3f} {flops/ms_torch*1e-9:<10.1f}")
print(f"{'CUDA naive (参考)':<20} {'~6.80':<12} {'~1.3':<10}")
print(f"\n差距: Triton 朴素 / cuBLAS = {ms_naive/ms_torch:.1f}x")

## 总结

| 概念 | 说明 |
|------|------|
| GEMM | C = A @ B, 深度学习的核心计算 |
| 2D Grid | pid_m × pid_n 覆盖输出矩阵 |
| K 循环 | 沿 K 维分块累加: acc += tl.dot(a_blk, b_blk) |
| 混合精度 | FP16 输入, FP32 累加, FP16 输出 |
| 性能瓶颈 | 全局内存冗余访问 |

## 练习

1. **不同块大小**: 测试 BLOCK=32, 64, 128 对性能的影响
2. **转置矩阵乘**: 实现 C = A^T @ B
3. **批量矩阵乘**: 实现 C[b] = A[b] @ B[b]，用 3D grid
4. **计算 FLOPs**: 验证 2MNK 的浮点运算数